# 03- Modelagem: prevendo a popularidade de jogos da Steam

**Pergunta:** dá para prever a faixa de `peak_ccu` (pico de jogadores simultâneos) a partir das características da página do jogo?

**Alvo principal:** classificação em 4 faixas (`0`, `1-20`, `21-500`, `500+`), com **F1-macro** como métrica, porque 77,7% dos jogos estão na faixa 0 e só 1,0% na faixa 500+ (a acurácia seria enganosa).
**Complemento:** regressão em `log1p(peak_ccu)`.

Foram escolhidos três modelos de classificação para o projeto:

- **Regressão Logística:** modelo utilizado como referência para classificação, com ajuste de pesos para lidar com o desbalanceamento das classes.
- **MLP:** rede neural com duas camadas ocultas, contendo 64 e 32 neurônios.
- **Random Forest:** conjunto de árvores de decisão, utilizando 200 árvores.

Também foi utilizado um **Baseline**, que sempre prevê a classe mais frequente. Ele serve como referência para verificar se os modelos conseguem superar uma previsão simples.


**Roteiro**
1. Carga dos dados e split estratificado
2. Pré-processamento
3. Tratamento do desbalanceamento
4. Validação cruzada (5 folds) no treino
5. Avaliação final no teste
6. Importância das variáveis
7. Regressão complementar
8. Discussão

> Antes de rodar: o 01 e o 02 precisam ter gerado `../data/processed/features.parquet`.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.float_format", "{:.3f}".format)
SEED = 42

In [ ]:
dados = pd.read_parquet("../data/processed/features.parquet")
if "AppID" in dados.columns:
    dados = dados.set_index("AppID")

ordem = ["0", "1-20", "21-500", "500+"]

# alvos e colunas de vazamentos nao podem entrar como preditoras
alvos = [c for c in ["peak_ccu", "log_ccu", "faixa_ccu"] if c in dados.columns]
vazamentos = ["positive", "negative", "recommendations", "estimated_owners", "n_tags"]

print("faixa_ccu nulos:", dados["faixa_ccu"].isna().sum())
y_cls = dados["faixa_ccu"].astype(str)
y_reg = dados["log_ccu"] if "log_ccu" in dados.columns else np.log1p(dados["peak_ccu"])

X = dados.drop(columns=alvos + [c for c in vazamentos if c in dados.columns])
X = X.drop(columns=[c for c in X.columns if c.startswith(("pct_pos", "num_reviews"))])
X = X.select_dtypes(exclude=["datetime", "datetimetz"])

# categóricas viram códigos (NaN fica NaN); bool vira int8 só aqui, para o sklearn
for c in X.select_dtypes(["category", "object", "string"]).columns:
    X[c] = X[c].astype("category").cat.codes.replace(-1, np.nan)
X = X.astype({c: "int8" for c in X.select_dtypes("bool").columns})
# tipos anuláveis do pandas (Int64, boolean) viram float, que o sklearn aceita com NaN
for c in X.columns:
    if pd.api.types.is_extension_array_dtype(X[c]):
        X[c] = X[c].astype("float64")

print("X:", X.shape)
print("Colunas não numéricas restantes:", list(X.select_dtypes(exclude="number").columns))
print(y_cls.value_counts(normalize=True).reindex(ordem).round(3))

In [ ]:
X_tr, X_te, y_tr, y_te, yr_tr, yr_te = train_test_split(
    X, y_cls, y_reg, test_size=0.2, stratify=y_cls, random_state=SEED
)
print("treino:", X_tr.shape, "| teste:", X_te.shape)
print(y_te.value_counts().reindex(ordem))